### This notebook, will be used to visualise and see the results of the backtesting

In [ ]:
from sql_init import connect_to_mysql, close_connection
import port_op
import ctypes
import pandas as pd
import numpy as np

In [ ]:
file_path = "sql_login.json"
DB_NAME = "asx_stock_data"
connection, mycursor = connect_to_mysql(file_path, DB_NAME)

if connection is None:
    print("Failed to connect to MySQL database.")
print("Connected")

# Get the price data
query = """
SELECT symbol, date, close
FROM stock_data
"""
mycursor.execute(query)
columns = ["symbol", "date", "close"]

price_df = pd.DataFrame(mycursor.fetchall(), columns=columns)

# Get the signals
query = """
SELECT *
FROM signals
"""
mycursor.execute(query)
columns = ["symbol", "date", "log_return", "momentum", "mom_rank", "z_momentum", 
           "z_reversion", "avg_dollar_vol", "ma_50", "ma_200", "close_location"]

signal_df = pd.DataFrame(mycursor.fetchall(), columns=columns)

In [ ]:
price_df['date'] = pd.to_datetime(price_df['date'])
signal_df['date'] = pd.to_datetime(signal_df['date'])

In [ ]:
first_date = (signal_df[signal_df["momentum"].notna()]
              .groupby(signal_df["date"].dt.to_period("M"))["date"]
              .min())

In [ ]:
first_date.head()

In [ ]:
rebalancing = {}
for date in first_date:
    start_date = date - pd.DateOffset(days=252)
    # pick ideal stocks
    recent_signal = signal_df[(signal_df["date"]<=date) &
                              (signal_df["date"]>=start_date)]
    recent_price = price_df[(price_df["date"]<=date) &
                            (price_df["date"]>=start_date)]
    symbols = port_op.pick_stocks(recent_signal, date)

    # Allocate using a method
    return_df = recent_signal[recent_signal["symbol"].isin(symbols)][["symbol", "date", "log_return"]]
    price_data = recent_price[recent_price["symbol"].isin(symbols)][["symbol", "date", "close"]]

    data = (
        price_data
        .sort_values("date")
        .pivot(
            index="date",
            columns="symbol",
            values="close"
        )
    )
    cov_matrix = port_op.calculateCovMatrix(data)

    weights = port_op.mpt_w(return_df, cov_matrix)
    rebalancing[date] = weights

In [ ]:
close_connection(connection)